In [ ]:
# GodQuestions_Crawling

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
from bs4 import BeautifulSoup
import time
import re
import pandas as pd
from datetime import date

## Crawling Code

In [ ]:
final=pd.DataFrame(columns=["crawling_date","big_title_far","Question_FAR","Answer_FAR","URL_FAR","Question_ENG","Answer_ENG","URL_ENG"])
print(final)

In [ ]:
# 1) main title -> click & title extracting
driver = webdriver.Chrome()
driver.get("https://www.gotquestions.org/Farsi/")

a_tags=driver.find_elements(By.TAG_NAME,"a")
print(len(a_tags))

In [ ]:
for i in range(1, 52): # 첫 번째 클릭 (대주제) - 1부터 52까지
    

    try:
        element = driver.find_element(By.XPATH, f'/html/body/main/section[1]/div/strong/a[{i}]') # 버튼의 xpath를 이용 (1, 2, 3, ...)
        big_title_far = driver.find_element(By.XPATH, f'/html/body/main/section[1]/div/strong/a[{i}]').text # 대주제도 데이터 프레임에 넣기
        driver.execute_script("arguments[0].click();", element)
        time.sleep(0.5)

    except NoSuchElementException:
            time.sleep(0.5)
            break

    for ii in range(1, 120): # 두 번째 클릭 (소주제) 

        # 소주제 개수는 대주제마다 달라서, range()로 일단 1부터 120까지 범위 정해놓고
        # 반복문을 실행할 수 없는 상황, 찾으려는 요소가 없어서 더 이상 작업할 수 없을 때 반복문 종료
        try:
            element = driver.find_element(By.XPATH, f'/html/body/main/section[1]/div/strong/a[{ii}]') # 버튼의 xpath를 이용 (1, 2, 3, ...)
            driver.execute_script("arguments[0].click();", element)
            time.sleep(0.5)
        
        # 만약 버튼이 없다면 초기 화면으로 돌아간 후 초기 반복문으로 이동
        except NoSuchElementException:
            driver.get("https://www.gotquestions.org/Farsi/")
            time.sleep(0.5)
            break
            
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')

        # crawling 하는 날짜 date 넣기
        crawling_date = date.today()
        # 제목(소주제)는 <h1></h1> 사이에 존재
        title_far = soup.find('h1').get_text(strip = True)

        # 현재 url
        url_far = driver.current_url
            
        # 답변 내용은 "جواب"과 "English"사이에 존재한다는 공통점
        content_str_far = str(soup.get_text(separator="\n"))
        start_far = content_str_far.find('جواب') + len('جواب') # 답변의 시작 지점
        end_far = content_str_far.find('English') # 답변의 끝 지점
        answer_html_far = content_str_far[start_far : end_far]
        answer_far = BeautifulSoup(answer_html_far, 'html.parser').get_text("\n", strip=True) # <>와 같은 태그 제거하여 텍스트만 남기기

        # ----------- #

        # English 클릭해서
        #element = driver.find_element(By.XPATH,"/html/body/main/section[1]/div/a")
        
        try :
            element = driver.find_element(By.LINK_TEXT, "English")
            driver.execute_script("arguments[0].click();", element)
            time.sleep(0.5)

        # English 없는 경우 발생...ㅎㄷㄷ
        except NoSuchElementException: 
             # 페르시아어 파일 저장
            new_row = pd.DataFrame({"crawling_date":[crawling_date], "big_title_far":[big_title_far],
                                    "Question_FAR": [title_far], "Answer_FAR": [answer_far], "URL_FAR": [url_far],
                                    "Question_ENG": "NA", "Answer_ENG": "NA", "URL_ENG": "NA"})
            final = pd.concat([final, new_row], ignore_index=True)
            driver.back()
            time.sleep(0.1)
            break

        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
                
        # 영어 내용 크롤링
        # 영어 제목(소주제) <h1></h1> 사이에 존재
        title_eng = soup.find('h1').get_text(strip = True)

        # 현재 url
        url_eng = driver.current_url
                
        # 답변 내용은 "Question"과 "Return to:" 사이에 존재한다는 공통점
        content_str_eng = str(soup.get_text(separator="\n"))
        start_eng = content_str_eng.find('Answer') + len('Answer') # 답변의 시작 지점
        end_eng = content_str_eng.find('Return to:') # 답변의 끝 지점
        answer_html_eng = content_str_eng[start_eng : end_eng]
        answer_eng = BeautifulSoup(answer_html_eng, 'html.parser').get_text("\n", strip=True) # <>와 같은 태그 제거하여 텍스트만 남기기

            
        # 파일 저장
        new_row = pd.DataFrame({"crawling_date":[crawling_date], "big_title_far":[big_title_far],
                                    "Question_FAR": [title_far], "Answer_FAR": [answer_far], "URL_FAR": [url_far],
                                    "Question_ENG": [title_eng], "Answer_ENG": [answer_eng], "URL_ENG": [url_eng]})
        final = pd.concat([final, new_row], ignore_index=True)
            
        driver.back()
        time.sleep(0.1)

        driver.back()
        time.sleep(0.1)

In [ ]:
final

In [ ]:
# 추후 자료 매칭을 위해
final["URL_FAR_dup"]=final["URL_FAR"].str.replace(r"https://www.gotquestions.org/Farsi/Farsi-", "", regex=True)
final["URL_FAR_dup"]=final["URL_FAR_dup"].str.replace(r".html", "", regex=True)

In [ ]:
final.to_excel(f"/Users/haley/Desktop/2025-1/DS1/code/preprocessing/GodQuestions/GodQuestions_raw_Farsi__{crawling_date}.xlsx",index=False)